### Imports

In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
from python_calamine import load_workbook

### Read the data from the Excel file

In [ ]:
RAW_PATH = Path("../data/raw")
assert RAW_PATH.exists()

In [ ]:
SIMULADOR_FILE = RAW_PATH / "Simulador Bloco e Subbloco_v2.xlsx"
assert SIMULADOR_FILE.exists()

In [ ]:
excel = pd.ExcelFile(SIMULADOR_FILE, engine="calamine")                     
print(excel.sheet_names)

# Data cleaning steps

1. Extract 'Calendário FV' sheet from the provided Excel file.

In [ ]:
# TODO: extract 'Calendário FV' sheet to csv

In [ ]:
# 1. Quick check for external workbook links in XLSX package relationships
with zipfile.ZipFile(SIMULADOR_FILE) as z:
    ext_links = [name for name in z.namelist() if "externalLink" in name]
    if ext_links:
        print(f"External workbook dependencies found: {ext_links}")
    else:
        print("No external workbook dependencies found in package metadata.")

In [ ]:
# 2. Fast scan for evaluated errors (#REF!, #VALUE!, #DIV/0!, #N/A, etc.) using Calamine
wb = load_workbook(SIMULADOR_FILE)
excel_error_tokens = {"#REF!", "#VALUE!", "#DIV/0!", "#NAME?", "#N/A", "#NUM!", "#NULL!"}
broken_items = []

In [ ]:
def col_to_letter(col_idx: int) -> str:
    """Convert a 0-indexed column integer to Excel column coordinate (e.g. 0 -> A, 27 -> AB)."""
    result = ""
    col_idx += 1
    while col_idx > 0:
        col_idx, remainder = divmod(col_idx - 1, 26)
        result = chr(65 + remainder) + result
    return result


for sheet_name in wb.sheet_names:
    ws = wb.get_sheet_by_name(sheet_name)
    for r_idx, row in enumerate(ws.to_python()):
        for c_idx, val in enumerate(row):
            if isinstance(val, str) and val in excel_error_tokens:
                coord = f"{col_to_letter(c_idx)}{r_idx + 1}"
                broken_items.append({
                    "sheet": sheet_name,
                    "cell": coord,
                    "issue_type": f"Cached error ({val})",
                    "error": val,
                })

In [ ]:

df_broken = pd.DataFrame(broken_items)

print(f"Total issues found: {len(df_broken)}")
if not df_broken.empty:
    print("\nSummary by sheet and issue type:")
    print(df_broken.groupby(["sheet", "issue_type"]).size().rename("count"))

df_broken.head(20)
